# ClimateGuard Fine-tuning with Unsloth

ClimateGuard is an offline disaster preparedness assistant built for climate-vulnerable, low-connectivity communities.

This notebook demonstrates a reproducible Gemma fine-tuning setup using Unsloth QLoRA (4-bit), aligned to the Gemma 4 Good Hackathon goals:
- social impact in real-world settings,
- technical execution with working systems,
- clear use case and practical utility.

Official competition page:
https://www.kaggle.com/competitions/gemma-4-good-hackathon

In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# 1. Load Model & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-31b-it-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 288/288 [00:01<00:00, 1.09s/it]
Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.
✅ Model loaded
GPU: 4.70 GB used
✅ LoRA applied


In [ ]:
# 3. Training Data Prompt
prompt = """<start_of_turn>user
{}
Context: {}<end_of_turn>
<start_of_turn>model
{}<end_of_turn>"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    contexts     = examples["context"]
    responses    = examples["response"]
    texts = []
    for instruction, context, response in zip(instructions, contexts, responses):
        text = prompt.format(instruction, context, response)
        texts.append(text)
    return { "text" : texts, }

# 4. Load Dataset
dataset = load_dataset("json", data_files="training_data.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

Unsloth: Tokenizing ["text"] (num_proc=8): 100% 20/20 [00:41<00:00, 2.24s/ examples]


In [ ]:
# 5. Trainer Setup
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

# 6. Start Training
trainer.train()

🚀 Training started safely...
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 3 | Total steps = 9
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
 [9/9 00:30, Epoch 3/3]
Step    Training Loss
5       1.201021
✅ Done! Loss: 1.1534


In [ ]:
# 7. Save the Fine-tuned Model for Ollama
model.save_pretrained_gguf("climateguard_gemma_31b", tokenizer, quantization_method = "q4_k_m")
print("Model saved and ready for Ollama deployment.")

print("\nSubmission alignment notes:")
print("- Impact: offline emergency support for climate-vulnerable communities")
print("- Technical execution: reproducible QLoRA fine-tuning pipeline")
print("- Clear use case: disaster triage and actionable survival guidance")